<a href="https://colab.research.google.com/github/rpreddy56789-hash/GoogleCollab/blob/feature%2Fpyspark_basics/sparksession.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from pyspark.sql import SparkSession

In [7]:
spark_session = SparkSession.builder.getOrCreate()

In [8]:
spark_session.version

'4.0.2'

In [9]:
import kagglehub
path = kagglehub.dataset_download("shivamb/netflix-shows")

100%|██████████| 1.34M/1.34M [00:00<00:00, 1.44MB/s]

Extracting files...


In [24]:
df = spark_session.read.csv(path, header=True, schema=df_schema)

In [16]:
df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [17]:
df.dtypes

[('show_id', 'string'),
 ('type', 'string'),
 ('title', 'string'),
 ('director', 'string'),
 ('cast', 'string'),
 ('country', 'string'),
 ('date_added', 'string'),
 ('release_year', 'string'),
 ('rating', 'string'),
 ('duration', 'string'),
 ('listed_in', 'string'),
 ('description', 'string')]

In [21]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

In [23]:
df_schema = StructType([
    StructField("show_id", StringType(), True),
    StructField("type", StringType(), True),
    StructField("title", StringType(), True),
    StructField("director", StringType(), True),
    StructField("cast", StringType(), True),
    StructField("country", StringType(), True),
    StructField("date_added", StringType(), True),
    StructField("release_year", IntegerType(), True),
    StructField("rating", StringType(), True),
    StructField("duration", StringType(), True),
    StructField("listed_in", StringType(), True),
    StructField("description", StringType(), True)
])

In [25]:
df.dtypes

[('show_id', 'string'),
 ('type', 'string'),
 ('title', 'string'),
 ('director', 'string'),
 ('cast', 'string'),
 ('country', 'string'),
 ('date_added', 'string'),
 ('release_year', 'int'),
 ('rating', 'string'),
 ('duration', 'string'),
 ('listed_in', 'string'),
 ('description', 'string')]

In [28]:
df_schema = spark_session.read.csv(path, header=True, schema=df_schema)

In [29]:
df_schema.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [36]:
df_count = df_schema.count()
print(df_count)

8809


In [37]:
df_filter = df.filter(df.type == "Movie")
df_filter.show()

+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|show_id| type|               title|            director|                cast|             country|        date_added|release_year|rating|duration|           listed_in|         description|
+-------+-----+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+--------+--------------------+--------------------+
|     s1|Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|  90 min|       Documentaries|As her father nea...|
|     s7|Movie|My Little Pony: A...|Robert Cullen, Jo...|Vanessa Hudgens, ...|                NULL|September 24, 2021|        2021|    PG|  91 min|Children & Family...|Equestria's divid...|
|     s8|Movie|             Sankofa|        Haile 

In [38]:
df_max = df.groupBy("country").count().orderBy("count", ascending=False)
df_max.show()

+--------------------+-----+
|             country|count|
+--------------------+-----+
|       United States| 2805|
|               India|  972|
|                NULL|  832|
|      United Kingdom|  419|
|               Japan|  245|
|         South Korea|  199|
|              Canada|  181|
|               Spain|  145|
|              France|  123|
|              Mexico|  110|
|               Egypt|  106|
|              Turkey|  105|
|             Nigeria|   93|
|           Australia|   87|
|              Taiwan|   81|
|           Indonesia|   79|
|              Brazil|   77|
|United Kingdom, U...|   75|
|         Philippines|   75|
|United States, Ca...|   73|
+--------------------+-----+
only showing top 20 rows


In [47]:
from pyspark.sql.functions import when, col, regexp_extract, lit

df_duration_time = df.withColumn(
    "duration_time",
    when(col("duration").contains("min"), regexp_extract(col("duration"), "(\\d+)\\smin", 1).cast(IntegerType()))
    .when(col("duration").contains("Season"), regexp_extract(col("duration"), "(\\d+)\\sSeason(s)?", 1).cast(IntegerType()) * lit(90))
    .otherwise(lit(None).cast(IntegerType()))
)
df_duration_time.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+-------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|duration_time|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+-------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|           90|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|A

In [48]:
df_duration_time.dtypes

[('show_id', 'string'),
 ('type', 'string'),
 ('title', 'string'),
 ('director', 'string'),
 ('cast', 'string'),
 ('country', 'string'),
 ('date_added', 'string'),
 ('release_year', 'int'),
 ('rating', 'string'),
 ('duration', 'string'),
 ('listed_in', 'string'),
 ('description', 'string'),
 ('duration_time', 'int')]

In [57]:
df_max_duration = df_duration_time.filter(df_duration_time.release_year.isNotNull()).groupBy("release_year","title").agg({"duration_time": "max"}).orderBy("max(duration_time)", ascending=False)
df_max_duration.show()

+------------+--------------------+------------------+
|release_year|               title|max(duration_time)|
+------------+--------------------+------------------+
|        2020|      Grey's Anatomy|              1530|
|        2019|        Supernatural|              1350|
|        2017|                NCIS|              1350|
|        2019|           Heartland|              1170|
|        2015|        Red vs. Blue|              1170|
|        2019|COMEDIANS of the ...|              1170|
|        2017|      Criminal Minds|              1080|
|        2018|   Trailer Park Boys|              1080|
|        2003|             Frasier|               990|
|        1992|              Cheers|               990|
|        2006|       Stargate SG-1|               900|
|        1992|Danger Mouse: Cla...|               900|
|        2003|             Friends|               900|
|        2019|    Shameless (U.S.)|               900|
|        2019|    The Walking Dead|               900|
|        1